# RAG (Retrieval-Augmented Generation) Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Document Chunking

In [ ]:
```python

def chunk_text(text, chunk_size=200, overlap=50):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
```

### Step 2: Hosted Embedding Call

In production, embedding is a single API call to a hosted model. The response is a fixed-dimension vector per chunk; the math (cosine similarity, top-k) below does not care which model produced the vector.

In [ ]:
```python

import os

import requests

EMBEDDING_URL = os.environ["EMBEDDING_URL"]    # gateway URL, e.g. https://llm.internal/v1/embeddings

EMBEDDING_MODEL = os.environ["EMBEDDING_MODEL"]  # e.g. text-embedding-3-small

def embed(texts):

    response = requests.post(

        EMBEDDING_URL,

        headers={"Authorization": f"Bearer {os.environ['GATEWAY_TOKEN']}"},

        json={"model": EMBEDDING_MODEL, "input": texts},

        timeout=30,

    )

    response.raise_for_status()

    return [item["embedding"] for item in response.json()["data"]]

In [ ]:
```

### Step 3: Cosine Similarity Search

In [ ]:
```python

def cosine_similarity(a, b):

    dot = sum(x * y for x, y in zip(a, b))

    norm_a = math.sqrt(sum(x * x for x in a))

    norm_b = math.sqrt(sum(x * x for x in b))

    if norm_a == 0 or norm_b == 0:

        return 0.0

    return dot / (norm_a * norm_b)

def search(query_embedding, stored_embeddings, top_k=5):

    scores = []

    for i, emb in enumerate(stored_embeddings):

        sim = cosine_similarity(query_embedding, emb)

        scores.append((i, sim))

    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:top_k]

In [ ]:
```

### Step 4: Prompt Construction

This is where the "augmented" in RAG happens. Take the retrieved chunks, format them into a prompt, and ask the LLM to answer based on the provided context.

In [ ]:
```python

def build_rag_prompt(query, retrieved_chunks):

    context = "\n\n---\n\n".join(

        f"[Source {i+1}]\n{chunk}"

        for i, chunk in enumerate(retrieved_chunks)

    )

    return f"""Answer the question based ONLY on the following context.

If the context doesn't contain enough information, say "I don't have enough information to answer that."

Context:

{context}

Question: {query}

Answer:"""

In [ ]:
```

### Step 5: The Complete RAG Pipeline

In [ ]:
```python

class RAGPipeline:

    def __init__(self):

        self.chunks = []

        self.embeddings = []

    def index(self, documents):

        all_chunks = []

        for doc in documents:

            all_chunks.extend(chunk_text(doc))

        self.chunks = all_chunks

        self.embeddings = embed(all_chunks)

    def query(self, question, top_k=5):

        query_emb = embed([question])[0]

        results = search(query_emb, self.embeddings, top_k)

        retrieved = [(self.chunks[i], score) for i, score in results]

        prompt = build_rag_prompt(

            question, [chunk for chunk, _ in retrieved]

        )

        return prompt, retrieved

In [ ]:
```

### Step 6: Generation (simulated)

In production, this is where you call the LLM API. For this lesson, we simulate generation by extracting the most relevant sentence from the retrieved context.

In [ ]:
```python

def simple_generate(prompt, retrieved_chunks):

    query_words = set(prompt.lower().split("question:")[-1].split())

    best_sentence = ""

    best_score = 0

    for chunk in retrieved_chunks:

        for sentence in chunk.split("."):

            sentence = sentence.strip()

            if not sentence:

                continue

            words = set(sentence.lower().split())

            overlap = len(query_words & words)

            if overlap > best_score:

                best_score = overlap

                best_sentence = sentence

    return best_sentence if best_sentence else "I don't have enough information."

In [ ]:
```

## Exercises

In [ ]:
1. Experiment with chunk sizes: try 50, 100, 200, and 500 words on the same document set. For each size, run the same 5 queries and count how many return a relevant chunk in the top-3. Find the sweet spot where retrieval quality peaks.

2. Add metadata to each chunk (source document name, chunk position). Modify the prompt template to include source attribution so the LLM cites its sources.

3. Implement a simple evaluation: given 10 question-answer pairs, run each question through the RAG pipeline, and measure what percentage of retrieved chunks contain the answer. This is retrieval recall at k.

5. Build a conversation-aware RAG pipeline: maintain a history of the last 3 exchanges and include them in the prompt alongside the retrieved chunks. Test with follow-up questions like "What about enterprise?" after asking about pricing.